
# V29 OFFICIAL — Per-Label Specialist Ensemble

**Goal: public Kaggle score ≥ 0.600**

V29 uses only the current official partition:

- Train: 3,292
- Validation: 500
- Test: 1,000

The validation set is **not used to train the neural models**. It is used only after train-only
5-fold OOF model/ensemble ranking is locked.

## Why V29 is different

V28 trained every label too similarly. V29 uses four complementary transformer experts:

1. **BanglaHateBERT** — explicit abuse / toxicity specialist
2. **Bengali-Abusive MuRIL** — Bengali abusive-language domain expert
3. **XLM-R** — semantic / implicit / sarcasm specialist
4. **BanglishBERT** — Romanized and code-mixed Bengali specialist

Plus:

5. **Character + word TF-IDF logistic regression** — lexical/spelling/censorship expert

### Label-specialist loss

The transformer models still output all seven labels, but each model receives a different
label-focus vector. For secondary attributes, non-toxic negatives are downweighted, so the model
learns to distinguish one type of toxicity from other toxic comments instead of being dominated by
easy non-toxic negatives.

No aggressive `class_weight="balanced"` is used.

### Calibration

The official train and validation prevalences are almost identical. Therefore V29 evaluates both:

- cross-fitted probability thresholds
- prevalence-constrained top-k ranking

Only a very small calibration menu is allowed on validation to reduce overfitting.

### Submission rule

The notebook creates a submission, but **send `v29_results.zip` to ChatGPT before Kaggle submission**.


In [ ]:

# 1. Upload V29 data
from google.colab import files
import pathlib, zipfile

uploaded = files.upload()
assert "v29_OFFICIAL_data.zip" in uploaded, "Upload v29_OFFICIAL_data.zip."

DATA_DIR = pathlib.Path("/content/v29_data")
DATA_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile("v29_OFFICIAL_data.zip") as z:
    z.extractall(DATA_DIR)

print("Extracted:")
for p in sorted(DATA_DIR.iterdir()):
    print(" ", p.name, f"{p.stat().st_size:,} bytes")


In [ ]:

# 2. Dependencies + checkpoint preflight
import sys, subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.6",
    "tokenizers==0.22.1",
    "sentencepiece==0.2.1",
    "scikit-learn",
    "scipy",
    "pandas",
    "numpy",
    "tqdm",
    "safetensors",
])

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/csebuetnlp/normalizer"
])

from transformers import AutoConfig, AutoTokenizer

MODEL_NAMES = {
    "banglahate": "saroarj/BanglaHateBert",
    "abusive_muril": "Hate-speech-CNERG/bengali-abusive-MuRIL",
    "xlmr": "FacebookAI/xlm-roberta-base",
    "banglish": "csebuetnlp/banglishbert",
}

for key, name in MODEL_NAMES.items():
    print("Checking:", key, name)
    _ = AutoConfig.from_pretrained(name)
    _ = AutoTokenizer.from_pretrained(name)

print("✅ All four checkpoints accessible.")


In [ ]:

# 3. Imports, settings, strict data verification
import gc, re, math, json, random, zipfile, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from scipy.sparse import hstack

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from normalizer import normalize

SEED = 2026
N_FOLDS = 5
EPOCHS = 3
SNAPSHOT_EPOCHS = [2,3]
MAX_LENGTH = 128
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10

LABELS = [
    "is_toxic","implicit_toxicity","sarcasm","contains_profanity",
    "threatening","identity_attack","sexual_content"
]
LI = {l:i for i,l in enumerate(LABELS)}

MODEL_NAMES = {
    "banglahate": "saroarj/BanglaHateBert",
    "abusive_muril": "Hate-speech-CNERG/bengali-abusive-MuRIL",
    "xlmr": "FacebookAI/xlm-roberta-base",
    "banglish": "csebuetnlp/banglishbert",
}

MODEL_CFG = {
    "banglahate":{
        "batch":8, "accum":2, "lr":1.2e-5, "head_lr":2.5e-5,
        "text_col":"bn_text",
        "focus":[1.00,0.25,0.25,1.00,1.20,1.00,1.10],
        "script_weight":{"native":1.15,"mixed":1.00,"roman":0.85},
    },
    "abusive_muril":{
        "batch":4, "accum":4, "lr":1.0e-5, "head_lr":2.0e-5,
        "text_col":"raw_text",
        "focus":[1.00,0.45,0.40,0.80,1.00,1.00,1.00],
        "script_weight":{"native":1.00,"mixed":1.10,"roman":1.10},
    },
    "xlmr":{
        "batch":8, "accum":2, "lr":1.2e-5, "head_lr":2.5e-5,
        "text_col":"raw_text",
        "focus":[0.70,1.25,1.25,0.30,0.35,0.40,0.50],
        "script_weight":{"native":1.00,"mixed":1.05,"roman":1.05},
    },
    "banglish":{
        "batch":8, "accum":2, "lr":1.2e-5, "head_lr":2.5e-5,
        "text_col":"raw_text",
        "focus":[0.60,1.00,1.00,0.50,0.45,0.55,0.60],
        "script_weight":{"native":0.80,"mixed":1.30,"roman":1.50},
    },
}

RUN_DIR = Path("/content/v29_output")
RUN_DIR.mkdir(exist_ok=True)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
assert torch.cuda.is_available(), "Enable a T4 GPU."

train = pd.read_csv(DATA_DIR/"train.csv", dtype={"ID":str})
valid = pd.read_csv(DATA_DIR/"validation.csv", dtype={"ID":str})
test = pd.read_csv(DATA_DIR/"test.csv", dtype={"ID":str})
sample = pd.read_csv(DATA_DIR/"sample_submission.csv", dtype={"ID":str})

assert len(train)==3292
assert len(valid)==500
assert len(test)==1000
assert test["ID"].tolist()==sample["ID"].tolist()
assert set(train.ID).isdisjoint(valid.ID)
assert set(train.ID).isdisjoint(test.ID)
assert set(valid.ID).isdisjoint(test.ID)

print("✅ Official 3292 / 500 / 1000 partition verified.")

prevalence = pd.DataFrame({
    "train":train[LABELS].mean(),
    "validation":valid[LABELS].mean(),
})
prevalence["abs_gap"] = (prevalence["train"]-prevalence["validation"]).abs()
display(prevalence.round(5))

# This is useful evidence for rank/prevalence calibration.
print("Maximum train-validation prevalence gap:",
      float(prevalence["abs_gap"].max()))


In [ ]:

# 4. Bengali-aware preprocessing and grouped train-only folds
BN_RE = re.compile(r"[\u0980-\u09FF]")
LAT_RE = re.compile(r"[A-Za-z]")

def basic_clean(x):
    x = unicodedata.normalize("NFKC", str(x))
    x = x.replace("\u200c"," ").replace("\u200d"," ")
    return re.sub(r"\s+"," ",x).strip()

def bn_clean(x):
    x = basic_clean(x)
    try:
        x = normalize(x)
    except Exception:
        pass
    return re.sub(r"\s+"," ",x).strip()

def group_key(x):
    return basic_clean(x).lower()

def script_type(x):
    s = str(x)
    bn = bool(BN_RE.search(s))
    lat = bool(LAT_RE.search(s))
    if bn and lat:
        return "mixed"
    if bn:
        return "native"
    return "roman"

for df in [train,valid,test]:
    df["raw_text"] = df["text"].map(basic_clean)
    df["bn_text"] = df["text"].map(bn_clean)
    df["group_key"] = df["text"].map(group_key)
    df["script"] = df["text"].map(script_type)

# Multilabel grouped split, TRAIN ONLY.
groups = train.groupby("group_key",sort=False).agg(
    size=("ID","size"),
    **{f"sum_{l}":(l,"sum") for l in LABELS}
).reset_index()

global_pos = np.array([train[l].sum() for l in LABELS],float)
total = float(len(train))
rarity = 1/np.sqrt(np.maximum(global_pos,1))
mat = groups[[f"sum_{l}" for l in LABELS]].to_numpy(float)

rng = np.random.default_rng(SEED)
priority = (mat*rarity).sum(1) + 0.005*groups["size"].to_numpy()
order = np.argsort(-(priority+rng.random(len(groups))*1e-8))

fold_pos = np.zeros((N_FOLDS,len(LABELS)))
fold_size = np.zeros(N_FOLDS)
assignment = {}

for idx in order:
    row = groups.iloc[idx]
    vec = np.array([row[f"sum_{l}"] for l in LABELS],float)
    size = float(row["size"])

    costs=[]
    for f in range(N_FOLDS):
        pp=fold_pos.copy()
        ss=fold_size.copy()
        pp[f]+=vec
        ss[f]+=size
        cost = (
            np.mean((pp/np.maximum(global_pos[None,:],1)-1/N_FOLDS)**2)
            + 0.30*np.mean((ss/total-1/N_FOLDS)**2)
        )
        costs.append(cost)

    best=int(np.argmin(costs))
    assignment[row["group_key"]]=best
    fold_pos[best]+=vec
    fold_size[best]+=size

train["fold"] = train["group_key"].map(assignment).astype(int)

assert train.groupby("group_key")["fold"].nunique().max()==1
assert set(train["fold"])==set(range(N_FOLDS))

print("Fold sizes:",train.groupby("fold").size().to_dict())
display(train.groupby("fold")[LABELS].sum())

print("Script distribution:")
display(pd.DataFrame({
    "train":train.script.value_counts(normalize=True),
    "validation":valid.script.value_counts(normalize=True),
    "test":test.script.value_counts(normalize=True),
}).fillna(0).round(4))


In [ ]:

# 5. Lexical expert first: char + word TF-IDF, TRAIN-only OOF
# This is also an independent signal for spelling variation and censored vocabulary.

from scipy.sparse import hstack

lex_oof = np.zeros((len(train),len(LABELS)),np.float32)
lex_valid_parts=[]
lex_test_parts=[]

C_BY_LABEL = {
    "is_toxic":2.0,
    "implicit_toxicity":1.5,
    "sarcasm":1.5,
    "contains_profanity":4.0,
    "threatening":2.0,
    "identity_attack":3.0,
    "sexual_content":2.0,
}

for fold in range(N_FOLDS):
    tr=train[train.fold!=fold]
    va=train[train.fold==fold]

    char_vec=TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(2,6),
        min_df=2,
        max_df=0.995,
        max_features=150000,
        sublinear_tf=True,
        dtype=np.float32,
    )
    word_vec=TfidfVectorizer(
        analyzer="word",
        ngram_range=(1,2),
        min_df=2,
        max_df=0.995,
        max_features=70000,
        sublinear_tf=True,
        dtype=np.float32,
        token_pattern=r"(?u)\b\w+\b",
    )

    Ctr=char_vec.fit_transform(tr.raw_text)
    Cva=char_vec.transform(va.raw_text)
    Cev=char_vec.transform(valid.raw_text)
    Cte=char_vec.transform(test.raw_text)

    Wtr=word_vec.fit_transform(tr.raw_text)
    Wva=word_vec.transform(va.raw_text)
    Wev=word_vec.transform(valid.raw_text)
    Wte=word_vec.transform(test.raw_text)

    Xtr=hstack([Ctr,Wtr],format="csr")
    Xva=hstack([Cva,Wva],format="csr")
    Xev=hstack([Cev,Wev],format="csr")
    Xte=hstack([Cte,Wte],format="csr")

    pv=np.zeros((len(va),len(LABELS)),np.float32)
    pe=np.zeros((len(valid),len(LABELS)),np.float32)
    pt=np.zeros((len(test),len(LABELS)),np.float32)

    for j,label in enumerate(LABELS):
        clf=LogisticRegression(
            C=C_BY_LABEL[label],
            solver="liblinear",
            max_iter=1200,
            random_state=SEED+fold,
        )
        clf.fit(Xtr,tr[label].to_numpy(int))

        pv[:,j]=clf.predict_proba(Xva)[:,1]
        pe[:,j]=clf.predict_proba(Xev)[:,1]
        pt[:,j]=clf.predict_proba(Xte)[:,1]

    lex_oof[va.index.to_numpy()]=pv
    lex_valid_parts.append(pe)
    lex_test_parts.append(pt)

    print(
        f"lex fold {fold} | char={Ctr.shape[1]:,} "
        f"word={Wtr.shape[1]:,} total={Xtr.shape[1]:,}"
    )

lex_valid=np.mean(lex_valid_parts,axis=0)
lex_test=np.mean(lex_test_parts,axis=0)

np.savez_compressed(
    RUN_DIR/"lexical_oof_valid_test.npz",
    oof=lex_oof,
    validation=lex_valid,
    test=lex_test,
)

print("✅ Lexical expert complete.")


In [ ]:

# 6. Transformer dataset + LABEL-SPECIALIST weighted loss
class TextDS(Dataset):
    def __init__(self,frame,tok,text_col,with_labels=True):
        self.texts=frame[text_col].astype(str).tolist()
        self.scripts=frame["script"].astype(str).tolist()
        self.y=frame[LABELS].to_numpy(np.float32) if with_labels else None
        self.tok=tok

    def __len__(self):
        return len(self.texts)

    def __getitem__(self,i):
        e=self.tok(
            self.texts[i],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
        )
        e["script_name"]=self.scripts[i]
        if self.y is not None:
            e["labels"]=self.y[i]
        return e

class Collator:
    def __init__(self,tok):
        self.tok=tok

    def __call__(self,items):
        scripts=[x.pop("script_name") for x in items]

        y=None
        if "labels" in items[0]:
            y=torch.tensor(
                np.stack([x.pop("labels") for x in items]),
                dtype=torch.float32
            )

        b=self.tok.pad(
            items,
            padding=True,
            pad_to_multiple_of=8,
            return_tensors="pt"
        )

        b["script_name"]=scripts
        if y is not None:
            b["labels"]=y
        return b

# Easy-negative weights. For explicit subtypes, a non-toxic sample is usually
# an easy negative. Toxic negatives are retained at full weight.
EASY_NEG = torch.tensor(
    [1.00, 0.22, 0.22, 0.08, 0.08, 0.08, 0.08],
    dtype=torch.float32
)

def compute_pos_boost(frame):
    pos=frame[LABELS].sum().to_numpy(float)
    neg=len(frame)-pos

    # Mild boost only. Much less aggressive than class_weight="balanced".
    b=(np.maximum(neg,1)/np.maximum(pos,1))**0.25
    b=np.clip(b,1.0,2.0)
    b[0]=1.0
    return torch.tensor(b,dtype=torch.float32)

def specialist_loss(logits,y,scripts,expert,train_frame):
    cfg=MODEL_CFG[expert]

    raw=torch.nn.functional.binary_cross_entropy_with_logits(
        logits.float(),
        y,
        reduction="none"
    )

    focus=torch.tensor(
        cfg["focus"],
        dtype=torch.float32,
        device=logits.device
    ).unsqueeze(0)

    easy=EASY_NEG.to(logits.device).unsqueeze(0)
    toxic=y[:,0:1]

    # Positive: full weight.
    # Negative attribute inside toxic comment: full weight (hard negative).
    # Negative attribute inside non-toxic comment: strongly reduced.
    neg_attr_weight=toxic + (1-toxic)*easy[:,1:]

    label_weight=torch.ones_like(raw)
    label_weight[:,1:]=torch.where(
        y[:,1:]>0.5,
        torch.ones_like(y[:,1:]),
        neg_attr_weight
    )

    # Mild rare-positive boost.
    pos_boost=compute_pos_boost(train_frame).to(logits.device).unsqueeze(0)
    label_weight=torch.where(
        y>0.5,
        label_weight*pos_boost,
        label_weight
    )

    # Script specialization.
    sw=torch.tensor(
        [cfg["script_weight"][s] for s in scripts],
        dtype=torch.float32,
        device=logits.device
    ).unsqueeze(1)

    weight=label_weight*focus*sw
    return (raw*weight).sum()/weight.sum().clamp_min(1e-8)

@torch.no_grad()
def predict(model,loader):
    model.eval()
    out=[]

    for b in loader:
        b.pop("labels",None)
        b.pop("script_name",None)
        inp={k:v.to("cuda",non_blocking=True) for k,v in b.items()}

        with torch.autocast(device_type="cuda",dtype=torch.float16):
            logits=model(**inp).logits

        out.append(torch.sigmoid(logits.float()).cpu().numpy())

    p=np.concatenate(out)
    assert np.isfinite(p).all()
    return p


In [ ]:

# 7. Train one transformer specialist with 5 train-only folds
fold_seeds=[2026,17,3407,1337,42]

def train_expert(expert):
    cfg=MODEL_CFG[expert]
    name=MODEL_NAMES[expert]
    text_col=cfg["text_col"]

    oof=np.zeros((len(train),len(LABELS)),np.float32)
    val_parts=[]
    test_parts=[]
    history=[]

    tok=AutoTokenizer.from_pretrained(name)

    for fold in range(N_FOLDS):
        tr=train[train.fold!=fold].copy()
        va=train[train.fold==fold].copy()

        assert set(tr.group_key).isdisjoint(set(va.group_key))

        cache=RUN_DIR/f"{expert}_fold_{fold}.npz"
        hp=RUN_DIR/f"{expert}_fold_{fold}_history.csv"

        if cache.exists():
            with np.load(cache) as z:
                idx=z["idx"]
                pv=z["valid"]
                pe=z["external_valid"]
                pt=z["test"]

            if np.array_equal(idx,va.index.to_numpy()):
                print("✅ reuse",expert,"fold",fold)
                oof[idx]=pv
                val_parts.append(pe)
                test_parts.append(pt)

                if hp.exists():
                    history.extend(pd.read_csv(hp).to_dict("records"))
                continue

        print("\n"+"="*74)
        print(expert,"fold",fold,"train",len(tr),"held",len(va))

        seed_everything(fold_seeds[fold])
        gc.collect()
        torch.cuda.empty_cache()

        col=Collator(tok)

        tl=DataLoader(
            TextDS(tr,tok,text_col,True),
            batch_size=cfg["batch"],
            shuffle=True,
            num_workers=0,
            pin_memory=True,
            collate_fn=col
        )
        vl=DataLoader(
            TextDS(va,tok,text_col,True),
            batch_size=32,
            shuffle=False,
            num_workers=0,
            pin_memory=True,
            collate_fn=col
        )
        evl=DataLoader(
            TextDS(valid,tok,text_col,False),
            batch_size=32,
            shuffle=False,
            num_workers=0,
            pin_memory=True,
            collate_fn=col
        )
        tel=DataLoader(
            TextDS(test,tok,text_col,False),
            batch_size=32,
            shuffle=False,
            num_workers=0,
            pin_memory=True,
            collate_fn=col
        )

        model=AutoModelForSequenceClassification.from_pretrained(
            name,
            num_labels=len(LABELS),
            problem_type="multi_label_classification",
            ignore_mismatched_sizes=True,
        ).to("cuda")

        no_decay=["bias","LayerNorm.weight","layer_norm.weight"]

        backbone_decay=[]
        backbone_nodecay=[]
        head=[]

        for n,p in model.named_parameters():
            if not p.requires_grad:
                continue

            if "classifier" in n or "score" in n:
                head.append(p)
            elif any(x in n for x in no_decay):
                backbone_nodecay.append(p)
            else:
                backbone_decay.append(p)

        groups=[]
        if backbone_decay:
            groups.append({
                "params":backbone_decay,
                "lr":cfg["lr"],
                "weight_decay":WEIGHT_DECAY
            })
        if backbone_nodecay:
            groups.append({
                "params":backbone_nodecay,
                "lr":cfg["lr"],
                "weight_decay":0.0
            })
        if head:
            groups.append({
                "params":head,
                "lr":cfg["head_lr"],
                "weight_decay":WEIGHT_DECAY
            })

        opt=torch.optim.AdamW(groups)

        steps=math.ceil(len(tl)/cfg["accum"])*EPOCHS
        sched=get_linear_schedule_with_warmup(
            opt,
            int(steps*WARMUP_RATIO),
            steps
        )
        scaler=torch.amp.GradScaler("cuda")

        snapshots_va=[]
        snapshots_ev=[]
        snapshots_te=[]
        fh=[]

        for epoch in range(1,EPOCHS+1):
            model.train()
            opt.zero_grad(set_to_none=True)

            running=0.0
            seen=0

            for step,b in enumerate(
                tqdm(tl,desc=f"{expert} f{fold} e{epoch}/{EPOCHS}")
            ):
                y=b.pop("labels").to("cuda",non_blocking=True)
                scripts=b.pop("script_name")
                inp={k:v.to("cuda",non_blocking=True) for k,v in b.items()}

                with torch.autocast(device_type="cuda",dtype=torch.float16):
                    logits=model(**inp).logits

                loss=specialist_loss(
                    logits,y,scripts,expert,tr
                )

                if not torch.isfinite(loss):
                    raise FloatingPointError(
                        f"Non-finite loss: {expert} fold {fold} epoch {epoch}"
                    )

                scaler.scale(loss/cfg["accum"]).backward()

                running+=float(loss.item())*len(y)
                seen+=len(y)

                if (step+1)%cfg["accum"]==0 or step+1==len(tl):
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)

                    old=scaler.get_scale()
                    scaler.step(opt)
                    scaler.update()

                    if scaler.get_scale()>=old:
                        sched.step()

                    opt.zero_grad(set_to_none=True)

            rec={
                "expert":expert,
                "fold":fold,
                "epoch":epoch,
                "loss":running/max(seen,1)
            }
            print(rec)
            fh.append(rec)
            history.append(rec)
            pd.DataFrame(fh).to_csv(hp,index=False)

            if epoch in SNAPSHOT_EPOCHS:
                snapshots_va.append(predict(model,vl))
                snapshots_ev.append(predict(model,evl))
                snapshots_te.append(predict(model,tel))

        pv=np.mean(snapshots_va,axis=0)
        pe=np.mean(snapshots_ev,axis=0)
        pt=np.mean(snapshots_te,axis=0)

        oof[va.index.to_numpy()]=pv
        val_parts.append(pe)
        test_parts.append(pt)

        np.savez_compressed(
            cache,
            idx=va.index.to_numpy(),
            valid=pv,
            external_valid=pe,
            test=pt,
        )

        del model,opt,sched,scaler,tl,vl,evl,tel
        gc.collect()
        torch.cuda.empty_cache()

    pval=np.mean(val_parts,axis=0)
    ptest=np.mean(test_parts,axis=0)

    np.savez_compressed(
        RUN_DIR/f"{expert}_oof_valid_test.npz",
        oof=oof,
        validation=pval,
        test=ptest
    )

    pd.DataFrame(history).to_csv(
        RUN_DIR/f"{expert}_training_history.csv",
        index=False
    )

    return oof,pval,ptest


In [ ]:

# 8. Train all four transformer specialists
# Long cell: 20 fold-models total. Completed folds are cached.

outputs={}

for expert in ["banglahate","abusive_muril","xlmr","banglish"]:
    print("\n\n"+"#"*80)
    print("TRAINING:",expert.upper())
    print("#"*80)

    outputs[expert]=train_expert(expert)

bh_oof,bh_val,bh_test=outputs["banglahate"]
mu_oof,mu_val,mu_test=outputs["abusive_muril"]
xl_oof,xl_val,xl_test=outputs["xlmr"]
bg_oof,bg_val,bg_test=outputs["banglish"]

print("✅ All transformer specialists complete.")


In [ ]:

# 9. Exact competition metrics and calibration helpers
y_train=train[LABELS].to_numpy(np.int8)
y_valid=valid[LABELS].to_numpy(np.int8)
folds=train["fold"].to_numpy(int)

def f1_pos(y,p):
    y=np.asarray(y,np.int8)
    p=np.asarray(p,np.int8)

    tp=np.sum((y==1)&(p==1))
    fp=np.sum((y==0)&(p==1))
    fn=np.sum((y==1)&(p==0))

    d=2*tp+fp+fn
    return float(2*tp/d) if d else 0.0

def f1_macro_binary(y,p):
    vals=[]
    y=np.asarray(y,np.int8)
    p=np.asarray(p,np.int8)

    for c in [0,1]:
        tp=np.sum((y==c)&(p==c))
        fp=np.sum((y!=c)&(p==c))
        fn=np.sum((y==c)&(p!=c))
        d=2*tp+fp+fn
        vals.append(2*tp/d if d else 0.0)

    return float(np.mean(vals))

def metric(label,y,p):
    return (
        f1_macro_binary(y,p)
        if label=="is_toxic"
        else f1_pos(y,p)
    )

def official_score(y,pred):
    scores=[
        metric(l,y[:,j],pred[:,j])
        for j,l in enumerate(LABELS)
    ]
    final=0.5*scores[0]+0.5*np.mean(scores[1:])
    return float(final),scores

def best_threshold_regularized(label,y,p):
    vals=np.unique(p)

    if len(vals)>600:
        vals=np.quantile(vals,np.linspace(0,1,600))

    cand=np.unique(np.r_[0.0,vals,1.000001])
    true_rate=float(np.mean(y))

    # Fixed mild prior penalty; no tuning.
    lam=0.03 if label=="is_toxic" else 0.08

    best=-1e9
    best_t=0.5

    for t in cand:
        pred=(p>=t).astype(np.int8)
        raw=metric(label,y,pred)
        adjusted=raw-lam*abs(float(pred.mean())-true_rate)

        if adjusted>best:
            best=adjusted
            best_t=float(t)

    return best_t

def crossfit_threshold(label,y,p):
    pred=np.zeros(len(y),np.int8)
    fs=[]
    ts=[]

    for f in range(N_FOLDS):
        held=folds==f

        t=best_threshold_regularized(
            label,y[~held],p[~held]
        )

        pred[held]=(p[held]>=t).astype(np.int8)
        ts.append(t)
        fs.append(metric(label,y[held],pred[held]))

    return {
        "prediction":pred,
        "score":metric(label,y,pred),
        "fold_scores":np.asarray(fs,float),
        "thresholds":np.asarray(ts,float),
    }

def crossfit_topk(label,y,p,scale):
    pred=np.zeros(len(y),np.int8)
    fs=[]

    for f in range(N_FOLDS):
        held=folds==f
        idx=np.where(held)[0]

        # Prior learned only from the other four training folds.
        prior=float(y[~held].mean())*scale
        k=int(round(prior*len(idx)))
        k=max(0,min(k,len(idx)))

        order=np.argsort(-p[idx])
        if k>0:
            pred[idx[order[:k]]]=1

        fs.append(metric(label,y[held],pred[held]))

    return {
        "prediction":pred,
        "score":metric(label,y,pred),
        "fold_scores":np.asarray(fs,float),
        "scale":scale,
    }

def logit(p):
    p=np.clip(np.asarray(p,float),1e-6,1-1e-6)
    return np.log(p/(1-p))

def sigmoid(z):
    return 1/(1+np.exp(-z))

print("Metric helpers ready.")


In [ ]:

# 10. Build a compact set of label-specific ensemble candidates

SOURCE_OOF={
    "bh":bh_oof,
    "mu":mu_oof,
    "xl":xl_oof,
    "bg":bg_oof,
    "lex":lex_oof,
}
SOURCE_VAL={
    "bh":bh_val,
    "mu":mu_val,
    "xl":xl_val,
    "bg":bg_val,
    "lex":lex_valid,
}
SOURCE_TEST={
    "bh":bh_test,
    "mu":mu_test,
    "xl":xl_test,
    "bg":bg_test,
    "lex":lex_test,
}

# Allowed sources reflect the specialist design.
ALLOWED={
    "is_toxic":["bh","mu","xl","bg","lex"],
    "implicit_toxicity":["xl","bg","mu","bh","lex"],
    "sarcasm":["xl","bg","mu","bh","lex"],
    "contains_profanity":["bh","mu","lex","xl","bg"],
    "threatening":["bh","mu","xl","lex","bg"],
    "identity_attack":["bh","mu","xl","lex","bg"],
    "sexual_content":["bh","mu","bg","xl","lex"],
}

def generate_weight_configs(label):
    src=ALLOWED[label]

    configs=[]

    # Singles
    for s in src:
        configs.append({s:1.0})

    # 75/25 and 50/50 pairs among top four sources.
    pair_src=src[:4]
    for i in range(len(pair_src)):
        for k in range(i+1,len(pair_src)):
            a,b=pair_src[i],pair_src[k]
            configs.append({a:0.75,b:0.25})
            configs.append({a:0.25,b:0.75})
            configs.append({a:0.50,b:0.50})

    # Controlled triples.
    for a,b,c in [
        (src[0],src[1],src[2]),
        (src[0],src[1],src[4]),
        (src[0],src[2],src[4]),
        (src[1],src[2],src[4]),
    ]:
        configs.append({a:0.50,b:0.25,c:0.25})
        configs.append({a:0.25,b:0.50,c:0.25})

    # Deduplicate.
    seen=set()
    out=[]
    for c in configs:
        key=tuple(sorted(c.items()))
        if key not in seen:
            seen.add(key)
            out.append(c)

    return out

def blend_prob(source_dict,j,weights,mode):
    names=list(weights)
    ws=np.array([weights[n] for n in names],float)
    ws=ws/ws.sum()

    arrays=[source_dict[n][:,j] for n in names]

    if mode=="prob":
        out=np.zeros_like(arrays[0],dtype=float)
        for w,a in zip(ws,arrays):
            out+=w*a
        return out

    z=np.zeros_like(arrays[0],dtype=float)
    for w,a in zip(ws,arrays):
        z+=w*logit(a)
    return sigmoid(z)

print({
    l:len(generate_weight_configs(l))
    for l in LABELS
})


In [ ]:

# 11. TRAIN-ONLY model ranking
# Keep only TOP 3 candidate model blends per label before touching validation.

train_rows=[]
top_train={}

for j,label in enumerate(LABELS):
    yy=y_train[:,j]
    candidates=[]

    for weights in generate_weight_configs(label):
        for mode in ["prob","logit"]:
            p=blend_prob(
                SOURCE_OOF,j,weights,mode
            )

            # Two fixed calibration families.
            th=crossfit_threshold(label,yy,p)

            policies=[
                ("threshold",None,th)
            ]

            for scale in [0.90,1.00,1.10]:
                tk=crossfit_topk(
                    label,yy,p,scale
                )
                policies.append(
                    ("topk",scale,tk)
                )

            for policy,scale,res in policies:
                mean=float(res["fold_scores"].mean())
                std=float(res["fold_scores"].std())
                pred_rate=float(res["prediction"].mean())
                truth_rate=float(yy.mean())

                # Stronger stability/prior preference than V28.
                robust=(
                    mean
                    -0.10*std
                    -0.08*abs(pred_rate-truth_rate)
                )

                rec={
                    "label":label,
                    "weights_json":json.dumps(weights,sort_keys=True),
                    "mode":mode,
                    "policy":policy,
                    "scale":scale,
                    "crossfit_score":float(res["score"]),
                    "mean_fold":mean,
                    "std_fold":std,
                    "truth_rate":truth_rate,
                    "pred_rate":pred_rate,
                    "robust_score":robust,
                    "median_threshold":(
                        float(np.median(res["thresholds"]))
                        if policy=="threshold"
                        else np.nan
                    ),
                    "fold_scores":"|".join(
                        f"{x:.6f}" for x in res["fold_scores"]
                    ),
                }

                train_rows.append(rec)
                candidates.append({
                    "rec":rec,
                    "prob":p,
                    "cv":res,
                })

    # Sort and keep distinct top 3 configs only.
    candidates=sorted(
        candidates,
        key=lambda x:(
            x["rec"]["robust_score"],
            x["rec"]["crossfit_score"],
            -x["rec"]["std_fold"]
        ),
        reverse=True
    )

    kept=[]
    seen=set()

    for c in candidates:
        r=c["rec"]
        key=(
            r["weights_json"],
            r["mode"],
            r["policy"],
            r["scale"]
        )

        if key in seen:
            continue

        seen.add(key)
        kept.append(c)

        if len(kept)==3:
            break

    top_train[label]=kept

train_search=pd.DataFrame(train_rows)
train_search.to_csv(
    RUN_DIR/"v29_train_only_search.csv",
    index=False
)

print("Top 3 train-only candidates per label:")
for l in LABELS:
    print("\n",l)
    display(pd.DataFrame(
        [x["rec"] for x in top_train[l]]
    ).round(5))


In [ ]:

# 12. ONE-SHOT validation selection among only the locked top-3 train candidates
# Bootstrap penalty reduces the chance that one lucky rare-label example wins.

BOOTSTRAPS=200
boot_rng=np.random.default_rng(2901)

def apply_policy(label,p,rec,target_n,reference_prior):
    if rec["policy"]=="threshold":
        return (
            p>=float(rec["median_threshold"])
        ).astype(np.int8)

    scale=float(rec["scale"])
    k=int(round(reference_prior*scale*target_n))
    k=max(0,min(k,target_n))

    pred=np.zeros(target_n,np.int8)
    if k>0:
        order=np.argsort(-p)
        pred[order[:k]]=1
    return pred

def bootstrap_std(label,y,pred,n=BOOTSTRAPS):
    vals=[]
    N=len(y)

    for _ in range(n):
        idx=boot_rng.integers(0,N,size=N)
        vals.append(
            metric(label,y[idx],pred[idx])
        )

    return float(np.std(vals))

validation_rows=[]
selected={}

for j,label in enumerate(LABELS):
    options=[]

    for c in top_train[label]:
        rec=c["rec"]
        weights=json.loads(rec["weights_json"])

        pv=blend_prob(
            SOURCE_VAL,j,weights,rec["mode"]
        )

        pred=apply_policy(
            label,
            pv,
            rec,
            len(valid),
            float(train[label].mean())
        )

        score=metric(
            label,
            y_valid[:,j],
            pred
        )
        bstd=bootstrap_std(
            label,
            y_valid[:,j],
            pred
        )

        # Validation robustness, not raw F1 alone.
        robust=score-0.20*bstd

        row={
            **rec,
            "validation_score":float(score),
            "validation_bootstrap_std":float(bstd),
            "validation_robust":float(robust),
            "validation_pred_rate":float(pred.mean()),
            "validation_true_rate":float(y_valid[:,j].mean()),
        }

        validation_rows.append(row)
        options.append({
            "rec":row,
            "validation_prob":pv,
            "validation_pred":pred,
        })

    # Choose robust validation winner among just 3 locked train candidates.
    options=sorted(
        options,
        key=lambda x:(
            x["rec"]["validation_robust"],
            x["rec"]["validation_score"],
            x["rec"]["robust_score"]
        ),
        reverse=True
    )

    selected[label]=options[0]

validation_search=pd.DataFrame(validation_rows)
validation_search.to_csv(
    RUN_DIR/"v29_validation_selection.csv",
    index=False
)

print("FINAL SELECTED CONFIGS:")
for l in LABELS:
    display(pd.DataFrame(
        [selected[l]["rec"]]
    ).round(5))


In [ ]:

# 13. Final untouched-validation score and test predictions

val_pred=np.zeros((len(valid),len(LABELS)),np.int8)
test_pred=np.zeros((len(test),len(LABELS)),np.int8)

# Combined train+validation prevalence is used only for TOP-K test count.
combined_prior=pd.concat(
    [train[LABELS],valid[LABELS]],
    ignore_index=True
).mean()

final_configs=[]

for j,label in enumerate(LABELS):
    chosen=selected[label]
    rec=chosen["rec"]
    weights=json.loads(rec["weights_json"])

    val_pred[:,j]=chosen["validation_pred"]

    pt=blend_prob(
        SOURCE_TEST,j,weights,rec["mode"]
    )

    test_pred[:,j]=apply_policy(
        label,
        pt,
        rec,
        len(test),
        float(combined_prior[label])
    )

    final_configs.append(rec)

validation_official,validation_label_scores=official_score(
    y_valid,val_pred
)

validation_report=pd.DataFrame({
    "label":LABELS,
    "validation_score":validation_label_scores,
    "true_rate":y_valid.mean(axis=0),
    "pred_rate":val_pred.mean(axis=0),
    "test_pred_rate":test_pred.mean(axis=0),
})
validation_report.to_csv(
    RUN_DIR/"v29_final_validation_report.csv",
    index=False
)

pd.DataFrame(final_configs).to_csv(
    RUN_DIR/"v29_final_config.csv",
    index=False
)

# Hard gate targeted at the user's goal.
SUBMIT_CANDIDATE = bool(
    validation_official >= 0.600
    and validation_label_scores[0] >= 0.72
)

summary={
    "version":"v29_official_per_label_specialists",
    "target_public_score":0.600,
    "validation_official_score":float(validation_official),
    "validation_toxic_macro_f1":float(validation_label_scores[0]),
    "validation_attribute_mean_f1":float(
        np.mean(validation_label_scores[1:])
    ),
    "SUBMIT_CANDIDATE":SUBMIT_CANDIDATE,
    "note":"Models trained only on train; validation used for controlled top-3 config calibration."
}

(RUN_DIR/"v29_summary.json").write_text(
    json.dumps(summary,indent=2)
)

submission=sample.copy()
for j,l in enumerate(LABELS):
    submission[l]=test_pred[:,j]

assert submission.ID.tolist()==test.ID.tolist()
assert submission[LABELS].isin([0,1]).all().all()

submission.to_csv(
    RUN_DIR/"submission_v29_target600.csv",
    index=False
)

print(json.dumps(summary,indent=2))
display(validation_report.round(5))

if SUBMIT_CANDIDATE:
    print("\n✅ V29 PASSED THE 0.600 OFFLINE GATE.")
    print("Send v29_results.zip to ChatGPT before Kaggle submission.")
else:
    print("\n❌ V29 did not pass the 0.600 offline gate.")
    print("Do not submit yet; send v29_results.zip to ChatGPT.")


In [ ]:

# 14. Package all V29 outputs
np.savez_compressed(
    RUN_DIR/"v29_probabilities.npz",
    lexical_oof=lex_oof,
    lexical_validation=lex_valid,
    lexical_test=lex_test,
    banglahate_oof=bh_oof,
    banglahate_validation=bh_val,
    banglahate_test=bh_test,
    abusive_muril_oof=mu_oof,
    abusive_muril_validation=mu_val,
    abusive_muril_test=mu_test,
    xlmr_oof=xl_oof,
    xlmr_validation=xl_val,
    xlmr_test=xl_test,
    banglish_oof=bg_oof,
    banglish_validation=bg_val,
    banglish_test=bg_test,
)

zip_path=Path("/content/v29_results.zip")

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:
    for p in RUN_DIR.iterdir():
        if p.is_file():
            z.write(p,arcname=p.name)

print("Created:",zip_path)

from google.colab import files
files.download(str(zip_path))
